# Часть 1. Проверка гипотезы в Python и составление аналитической записки

Мы предобработали данные в SQL, и теперь они готовы для проверки гипотезы в Python. 

Напомним, как выглядит гипотеза: пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы. Попробуйте статистически это доказать, используя одностороннюю проверку гипотезы с двумя выборками:

Нулевая гипотеза $H_0: \mu_{\text{СПб}} \leq \mu_{\text{Москва}}$ <br> Среднее время активности пользователей в Санкт-Петербурге не больше, чем в Москве.

Альтернативная гипотеза $H_1: \mu_{\text{СПб}} > \mu_{\text{Москва}}$ <br> Среднее время активности пользователей в Санкт-Петербурге больше, и это различие статистически значимо.

По результатам анализа данных подготовим аналитическую записку, в которой опишем:

Выбранный тип t-теста и уровень статистической значимости.

Результат теста, или p-value.

Вывод на основе полученного p-value, то есть интерпретацию результатов.

Одну или две возможные причины, объясняющие полученные результаты.

## Проверка гипотезы и составление аналитической записки. Кейс Яндекс-книги

- Автор: Кропотов Валентин
- Дата: 29.11.2025

## Цели и задачи проекта


Цель проекта - проверить гипотезу о том, что пользователи из Санкт-Петербурга проводят больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы.

## Описание данных


В датасете `yandex_knigi_data` три поля:

- `city` - город географического положения. В нашем случае это `Москва` и `Санкт-Петербург`.
- `puid` - идентификатор пользователя.
- `hours` - суммарное количество часов чтения или прослушивания.

Данные представлены за период с 1 сентября по 11 декабря 2024 года.

## Содержимое проекта


1. Загрузка данных и знакомство с ними.
2. Проверка гипотезы.
3. Аналитическая записка.

---

## 1. Загрузка данных и знакомство с ними


In [1]:
# Импортируем библиотеки
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import ttest_ind
from statsmodels.stats.proportion import proportions_ztest

In [2]:
# Загружаем данные и смотрим первые пять строк
df = pd.read_csv('yandex_knigi_data.csv')

df.head()

,Unnamed: 0,city,puid,hours
0,0,Москва,9668,26.167776
1,1,Москва,16598,82.111217
2,2,Москва,80401,4.656906
3,3,Москва,140205,1.840556
4,4,Москва,248755,151.326434


- Столбец `Unnamed: 0`, который дублирует индексы, можно удалить.

In [3]:
# Удаляем столбец с заменой датафрейма
df.drop(columns=['Unnamed: 0'], inplace=True)

In [4]:
# Смотрим количество строк и пропуски
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8784 entries, 0 to 8783
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   city    8784 non-null   object 
 1   puid    8784 non-null   int64  
 2   hours   8784 non-null   float64
dtypes: float64(1), int64(1), object(1)
memory usage: 206.0+ KB


- В датасете 8784 строк без пропусков.

In [6]:
# Смотрим дубликаты
df[df.duplicated(subset='puid',keep='first')]

,city,puid,hours
6247,Санкт-Петербург,2637041,3.883926
6274,Санкт-Петербург,9979490,1.302997
6279,Санкт-Петербург,10597984,9.041320
6283,Санкт-Петербург,10815097,0.323291
6300,Санкт-Петербург,13626259,1.648434
...,...,...,...
8771,Санкт-Петербург,1130000018516717,0.517778
8772,Санкт-Петербург,1130000018954257,33.583294
8773,Санкт-Петербург,1130000020425037,2.386944
8775,Санкт-Петербург,1130000023864516,14.384722


- В данных 244 дубликата по полю `puid`. Видимо столбец `city` заполняется по геолокации использования приложения. Т.е. мы не можем точно установить родной город пользователя. Предлагаю оставить строки с количеством часов для Санкт-Петербурга.

In [7]:
df.drop_duplicates(subset='puid', keep='last', inplace=True)

## 2. Проверка гипотезы в Python

Гипотеза звучит так: пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы. Попробуем статистически это доказать, используя одностороннюю проверку гипотезы с двумя выборками:

- Нулевая гипотеза H₀: Средняя активность пользователей в часах в двух группах (Москва и Санкт-Петербург) не различается.

- Альтернативная гипотеза H₁: Средняя активность пользователей в Санкт-Петербурге больше, и это различие статистически значимо.

In [8]:
# Собираем данные в группы
msk = df[df['city'] == 'Москва']['hours']

spb = df[df['city'] == 'Санкт-Петербург']['hours']

# Посмотрим сколько в каждой группе пользователей
print(f'Количество пользователей из Москвы: {len(msk)}.\nКоличество пользователей из Санкт-Петербурга: {len(spb)}.')

Количество пользователей из Москвы: 5990.
Количество пользователей из Санкт-Петербурга: 2550.


Можно чуть детальнее рассмотреть выборки:

In [9]:
msk.describe()

count    5990.000000
mean       10.848192
std        36.925622
min         0.000022
25%         0.057042
50%         0.888232
75%         5.933439
max       857.209373
Name: hours, dtype: float64

In [10]:
spb.describe()

count    2550.000000
mean       11.592691
std        39.704993
min         0.000025
25%         0.080002
50%         0.984781
75%         6.509072
max       978.764775
Name: hours, dtype: float64

In [11]:
print(f'''
Сравнение 95-х квантилей в выборках:
Москва      Санкт-Петербург
{round(msk.quantile(0.95),2)}       {round(spb.quantile(0.95),2)}
''')


Сравнение 95-х квантилей в выборках:
Москва      Санкт-Петербург
54.93       57.33



Данные в группах очень неравномерные с большой разницей между средним и медианой из-за выбросов. При этом можно наблюдать разницу в показателях в пользу группы из Санкт-Петербурга. Будет ли эта разница статистически значима - вопрос для A/B-теста.

In [12]:
# Посчитаем дисперсию в выборках
var_spb, var_msk = np.var(spb), np.var(msk)
print(f'Дисперсия московской выборки: {round(var_msk,2)}.\nДисперсия выборки пользователей из Санкт-Петербурга: {round(var_spb,2)}.')

Дисперсия московской выборки: 1363.27.
Дисперсия выборки пользователей из Санкт-Петербурга: 1575.87.


- В группах достаточно большое количество наблюдений.
- Дисперсия в выборках различается, хоть и не сильно.
- T-тест Уэлча тут подойдет, потому что он предназначен для работы с выборками с разной дисперсией.
- Уровень значимости стандартный: 0.05.

In [13]:
# Задаем уровень значимости и проводим t-тест Уэлча
alpha = 0.05
stat_welsh_ttest, p_value_welsh_ttest = ttest_ind(spb,
                                                  msk,
                                                  equal_var=False,
                                                  alternative='greater')
if p_value_welsh_ttest>alpha:
    print(f'''p-value теста Уэлча: {round(p_value_welsh_ttest,3)}.
Нет оснований отвергать нулевую гипотезу.''')
else:
    print(f'''p-value теста Уэлча: {round(p_value_welsh_ttest,3)}.
Принимаем альтернативную гипотезу.''')

p-value теста Уэлча: 0.209.
Нет оснований отвергать нулевую гипотезу.


## 3. Аналитическая записка

- В выборках достаточно наблюдений для формирования нормального распределения выборочных средних и есть разница в дисперсиях, поэтому был выбран `t-тест Уэлча`. Уровень статистической значимости стандартный для такого рода тестов: `0.05`.
- В результате теста `p-value` получилось равным `0.209`.
- Полученное значение p-value `0.209` при уровне значимости `0.05` говорит о том, что у нас нет оснований отвергать нулевую гипотезу.
```
 Нулевая гипотеза H₀: Средняя активность пользователей в часах в двух группах (Москва и Санкт-Петербург) не различается.

 Альтернативная гипотеза H₁: Средняя активность пользователей в Санкт-Петербурге больше, и это различие статистически значимо.
```
    Таким образом получается, что мы не можем с уверенностью заявить, что пользователи из Санкт-Петербурга статистически значимо активнее пользователей из Москвы.

- Возможные причины такого результата:
    1. **Реальной разницы в активности пользователей по этим городам нет.**
      
       Значение p-value 0.209 говорит о том, что если бы нулевая гипотеза была верна, вероятность получить такие или еще более выраженные данные составляет 20.9%. Это довольно высокая вероятность, поэтому можно сделать вывод, что наблюдаемая небольшая разница в этих выборках могла возникнуть просто случайно, а не из-за реального различия между городами.
    2. **Недостаточный размер выборки или высокая дисперсия данных**

       Поскольку детектируемых эффект зависит в том числе от размера выборок и дисперсии в них, можно предополжить, что эффект есть, но он меньше минимального детектируемого эффекта нашего эксперимента. Если увеличить выборки и/или уменьшить дисперсию, мы сможем задетектировать более маленькие различия.

----

# Часть 2. Анализ результатов A/B-тестирования

Теперь вам нужно проанализировать другие данные. Представьте, что к вам обратились представители интернет-магазина BitMotion Kit, в котором продаются геймифицированные товары для тех, кто ведёт здоровый образ жизни. У него есть своя целевая аудитория, даже появились хиты продаж: эспандер со счётчиком и напоминанием, так и подстольный велотренажёр с Bluetooth.

В будущем компания хочет расширить ассортимент товаров. Но перед этим нужно решить одну проблему. Интерфейс онлайн-магазина слишком сложен для пользователей — об этом говорят отзывы.

Чтобы привлечь новых клиентов и увеличить число продаж, владельцы магазина разработали новую версию сайта и протестировали его на части пользователей. По задумке, это решение доказуемо повысит количество пользователей, которые совершат покупку.

Ваша задача — провести оценку результатов A/B-теста. В вашем распоряжении:

* данные о действиях пользователей и распределении их на группы,

* техническое задание.

Оцените корректность проведения теста и проанализируйте его результаты.

## 1. Опишите цели исследования.



Цель исследования - оценить корректность проведения A/B-теста и провести анализ его результатов.

## 2. Загрузите данные, оцените их целостность.


In [14]:
participants = pd.read_csv('ab_test_participants.csv')
events = pd.read_csv('ab_test_events.zip',
                     parse_dates=['event_dt'], low_memory=False)

- Оценим целостность данных в датасетах:

In [15]:
participants.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14525 entries, 0 to 14524
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   user_id  14525 non-null  object
 1   group    14525 non-null  object
 2   ab_test  14525 non-null  object
 3   device   14525 non-null  object
dtypes: object(4)
memory usage: 454.0+ KB


In [16]:
participants.head()

,user_id,group,ab_test,device
0,0002CE61FF2C4011,B,interface_eu_test,Mac
1,001064FEAAB631A1,B,recommender_system_test,Android
2,001064FEAAB631A1,A,interface_eu_test,Android
3,0010A1C096941592,A,recommender_system_test,Android
4,001E72F50D1C48FA,A,interface_eu_test,Mac


In [17]:
# Смотрим дубликаты по полям user_id и ab_test
participants.duplicated(subset=['user_id', 'ab_test']).sum()

0

- Датасет `participants` имеет 4 поля и 14 525 строк без пропусков и дубликатов по полям `user_id` и `ab_test`.

In [18]:
events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 787286 entries, 0 to 787285
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   user_id     787286 non-null  object        
 1   event_dt    787286 non-null  datetime64[ns]
 2   event_name  787286 non-null  object        
 3   details     249022 non-null  object        
dtypes: datetime64[ns](1), object(3)
memory usage: 24.0+ MB


In [19]:
events.head()

,user_id,event_dt,event_name,details
0,GLOBAL,2020-12-01 00:00:00,End of Black Friday Ads Campaign,ZONE_CODE15
1,CCBE9E7E99F94A08,2020-12-01 00:00:11,registration,0.0
2,GLOBAL,2020-12-01 00:00:25,product_page,NaN
3,CCBE9E7E99F94A08,2020-12-01 00:00:33,login,NaN
4,CCBE9E7E99F94A08,2020-12-01 00:00:52,product_page,NaN


In [20]:
events.duplicated().sum()

36318

In [21]:
events.drop_duplicates(inplace=True)

- Датасет `events` имеет 4 поля и 787 286 строк с пропусками в поле `details`. Всего обнаружено 36 318 дубликатов по всем столбцам. Можно их удалить.

## 3. По таблице `ab_test_participants` оценим корректность проведения теста:

   3\.1 Выделим пользователей, участвующих в тесте, и проверим:

   - соответствие требованиям технического задания,

   - равномерность распределения пользователей по группам теста,

   - отсутствие пересечений с конкурирующим тестом (нет пользователей, участвующих одновременно в двух тестовых группах).

In [22]:
# Отфильтровываем данные по нужному тесту
participants_test = participants[participants['ab_test'] == 'interface_eu_test']

# Считаем пользователей в группах
users_count = participants_test.groupby('group').agg({'user_id':'nunique'})
users_count

,user_id
group,
A,5383
B,5467


In [23]:
# Разница в граппах в процентном отношении
P = 100*(abs(users_count.iloc[0,0]-users_count.iloc[1,0]))/users_count.iloc[0,0]
print(f'Процентная разница в количестве пользователей в группах состовляет {P:.2f}%')

Процентная разница в количестве пользователей в группах состовляет 1.56%


- Группы почти не отличаются по размеру.

In [24]:
# Сравниваем группы по категориальной переменной - типу устройств
participants_test.groupby(['device', 'group']).agg({'user_id': 'nunique'}).unstack().sort_values(by=('user_id','A'), ascending=False)

user_id      
group         A     B
device               
Android    2445  2414
PC         1346  1418
iPhone     1026  1081
Mac         566   554

- Пользователи достаточно равномерно распределены по единственной категориальной переменной, которая нам доступна - типу устройства.

In [25]:
# Собираем id пользователей по группам
group_a = participants_test[participants_test['group'] == 'A']['user_id']
group_b = participants_test[participants_test['group'] == 'B']['user_id']

# Список пользователей, которые встречаются в обеих группаъ
list(set(group_a) & set(group_b))

[]

- Пользователей, которые встречались бы в обеих группах не обнаружено.

In [26]:
# Проследим, чтобы пользователи не пересекались между тестами
test_1 = participants[(participants['ab_test']=='interface_eu_test') & (participants['group']=='B')]['user_id']
test_2 = participants[(participants['ab_test']=='recommender_system_test') & (participants['group']=='B')]['user_id']

intersection = list(set(test_1) & set(test_2))

participants_test = participants_test[~participants_test['user_id'].isin(intersection)]

3\.2 Проанализируйте данные о пользовательской активности по таблице `ab_test_events`:

- оставьте только события, связанные с участвующими в изучаемом тесте пользователями;

In [27]:
# Серия с id пользователей, участвующих в тесте
test_users = participants_test['user_id']

# Оставляем в датафрейме только этих пользователй и проверяем
events_test = events[events['user_id'].isin(test_users)]
events_test['user_id'].nunique() == len(test_users)

True

- определим горизонт анализа: рассчитаем время (лайфтайм) совершения события пользователем после регистрации и оставьте только те события, которые были выполнены в течение первых семи дней с момента регистрации;

In [28]:
# Узнаем какие события есть в датасете
events_test['event_name'].unique()

array(['registration', 'login', 'product_page', 'purchase',
       'product_cart'], dtype=object)

In [29]:
# Сделаем отдельный датафрейм с полями user_id и датой регистрации
reg_dates = events_test[events_test['event_name'] == 'registration'][['user_id', 'event_dt']].rename(columns={'event_dt': 'reg_dt'})

# Объединим датафреймы для того, чтобы в каждой записи у каждого пользователя была дата регистрации
events_test = pd.merge(events_test, reg_dates, on='user_id', how='left')

# Создаем новый столбец с разницей в дате события и датой регистрации
events_test['days_since_reg'] = (events_test['event_dt'] - events_test['reg_dt']).dt.days

# Отфильтровываем по заданному параметру - первые семь дней после регистрации
events_filtered = events_test[events_test['days_since_reg'] <= 6]
events_filtered

,user_id,event_dt,event_name,details,reg_dt,days_since_reg
0,5F506CEBEDC05D30,2020-12-06 14:10:01,registration,0.0,2020-12-06 14:10:01,0
1,51278A006E918D97,2020-12-06 14:37:25,registration,-3.8,2020-12-06 14:37:25,0
2,A0C1E8EFAD874D8B,2020-12-06 17:20:22,registration,-3.32,2020-12-06 17:20:22,0
3,275A8D6254ACF530,2020-12-06 19:36:54,registration,-0.48,2020-12-06 19:36:54,0
4,0B704EB2DC7FCA4B,2020-12-06 19:42:20,registration,0.0,2020-12-06 19:42:20,0
...,...,...,...,...,...,...
72784,E89AF4EFC757D283,2020-12-29 21:46:43,product_cart,NaN,2020-12-23 09:35:48,6
72787,E89AF4EFC757D283,2020-12-29 21:47:56,product_cart,NaN,2020-12-23 09:35:48,6
72853,A6AFDC94A0D3B23D,2020-12-29 22:47:00,product_page,NaN,2020-12-23 13:53:33,6
72857,A6AFDC94A0D3B23D,2020-12-29 22:48:46,product_page,NaN,2020-12-23 13:53:33,6


Можно удалить вспомогательные столбцы

In [30]:
events_filtered = events_filtered.drop(columns=['reg_dt', 'days_since_reg'])
events_filtered

,user_id,event_dt,event_name,details
0,5F506CEBEDC05D30,2020-12-06 14:10:01,registration,0.0
1,51278A006E918D97,2020-12-06 14:37:25,registration,-3.8
2,A0C1E8EFAD874D8B,2020-12-06 17:20:22,registration,-3.32
3,275A8D6254ACF530,2020-12-06 19:36:54,registration,-0.48
4,0B704EB2DC7FCA4B,2020-12-06 19:42:20,registration,0.0
...,...,...,...,...
72784,E89AF4EFC757D283,2020-12-29 21:46:43,product_cart,NaN
72787,E89AF4EFC757D283,2020-12-29 21:47:56,product_cart,NaN
72853,A6AFDC94A0D3B23D,2020-12-29 22:47:00,product_page,NaN
72857,A6AFDC94A0D3B23D,2020-12-29 22:48:46,product_page,NaN


Так же можно добавить столбец с обозначением группы, в которую входит пользователь.

In [31]:
# Добавляем новый столбец с обозначением группы
events_filtered['group'] = 'A'

# Меняем значение для пользователей из группы B и проверяем
events_filtered.loc[events_filtered['user_id'].isin(group_b), 'group'] = 'B'
events_filtered.groupby('group').agg({'user_id':'nunique'})

,user_id
group,
A,5383
B,5351


Оцените достаточность выборки для получения статистически значимых результатов A/B-теста. Заданные параметры:

- базовый показатель конверсии — 30%,

- мощность теста — 80%,

- достоверность теста — 95%.

In [32]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

# Задаём параметры
alpha = 0.05  # Уровень значимости 1 - достоверность
power = 0.8  # Мощность теста
beta = 1 - power  # Ошибка второго рода, часто 1 - мощность
p = 0.3 # Базовый уровень доли
mde = 0.03  # Минимальный детектируемый эффект
effect_size = proportion_effectsize(p, p + mde)

# Инициализируем класс NormalIndPower
power_analysis = NormalIndPower()

# Рассчитываем размер выборки
sample_size = power_analysis.solve_power(
    effect_size = effect_size,
    power = power,
    alpha = alpha,
    ratio = 1 # Равномерное распределение выборок
)

print(f"Необходимый размер выборки для каждой группы: {int(sample_size)}")

Необходимый размер выборки для каждой группы: 3761


- Необходимо `3761` в каждой группе. У нас есть `5383` в группе А и `5351` в группе B.
- Для MDE в три процентных пункта у нас достаточное количество наблюдений.

- рассчитаем для каждой группы количество посетителей, сделавших покупку, и общее количество посетителей.

In [33]:
# Сделаем две таблицы: одну с уникальными пользователями, другую с уникальными пользователями, совершившими покупку
unique_users = events_filtered.groupby('group').agg({'user_id':'nunique'})
purchasers = events_filtered[events_filtered['event_name'] == 'purchase'].groupby('group').agg({'user_id':'nunique'})

# Объедимин таблицы по индексам
cr_table = unique_users.join(purchasers, lsuffix = '_unique', rsuffix = '_purch')
cr_table

,user_id_unique,user_id_purch
group,,
A,5383,1480
B,5351,1579


- В `группе A` `5383` уникальных пользователя, из которых `1480` совершили покупку в течении семи дней после регистрации.
- В `группе B` `5351` уникальных пользователя, из которых `1579` совершили покупку в течении семи дней после регистрации.

- сделаем предварительный общий вывод об изменении пользовательской активности в тестовой группе по сравнению с контрольной.

In [34]:
# Расчитаем CR в покупку в новом столбце
cr_table['cr'] = cr_table['user_id_purch'] / cr_table['user_id_unique']
cr_table

,user_id_unique,user_id_purch,cr
group,,,
A,5383,1480,0.274940
B,5351,1579,0.295085


In [35]:
print(f'''
Разница CR в покупку между тестовой и контрольной группами составляет {(cr_table.iloc[1,2]-cr_table.iloc[0,2])*100:.2f}% в пользу тестовой группы.
''')


Разница CR в покупку между тестовой и контрольной группами составляет 2.01% в пользу тестовой группы.



## 4. Проведите оценку результатов A/B-тестирования:

- Проверьте изменение конверсии подходящим статистическим тестом, учитывая все этапы проверки гипотез.

- **Нулевая гипотеза:**

    CR в покупку в двух группах не отличается.
- **Альтернативная гипотеза:**

    CR в покупку в группе B *больше*, чем в группе A.

In [36]:
# Подготовим данные для теста - добавим бинарный столбец
participants_test['purchaser'] = 0

# Собираем id покупателей
purch_users_id = events_filtered[events_filtered['event_name'] == 'purchase']['user_id'].unique()

# Дополняем бинарный столбец и проверяем результат
participants_test.loc[participants_test['user_id'].isin(purch_users_id), 'purchaser'] = 1
participants_test.groupby('group').agg({'purchaser':['count', 'sum', 'mean']})

purchaser                
          count   sum      mean
group                          
A          5383  1480  0.274940
B          5351  1579  0.295085

In [37]:
# Поскольку наша метрика - конверсия, будем использовать Z-тест пропорций
# Считаем количество наблюдений и успехов в каждой группе
n_a, n_b = participants_test[participants_test['group']=='A'].shape[0], participants_test[participants_test['group']=='B'].shape[0]
m_a = participants_test[participants_test['group']=='A']['purchaser'].sum()
m_b = participants_test[participants_test['group']=='B']['purchaser'].sum()

# Проверяем достаточность размера выборок
p_a, p_b = m_a/n_a, m_b/n_b

if (p_a*n_a > 10)and((1-p_a)*n_a > 10)and(p_b*n_b > 10)and((1-p_b)*n_b > 10):
    print('Предпосылка о достаточном количестве данных выполняется!')
else:
    print('Предпосылка о достаточном количестве данных НЕ выполняется!')

Предпосылка о достаточном количестве данных выполняется!


In [38]:
# Проведем Z-тест пропорций
alpha = 0.05
stat_ztest, p_value_ztest = proportions_ztest([m_b, m_a],
                                              [n_b, n_a],
                                              alternative='larger')
print(f'Значение p-value: {p_value_ztest} при уровне значимости: {alpha}.')

Значение p-value: 0.010393282955333764 при уровне значимости: 0.05.


**Выводы:**
- В проведении A/B-теста проблем не было обнаружено. В выборках достаточное количество наблюдений, распределение пользователей по группам достаточно репрезентативное, по крайней мере по тем данным, что у нас есть.
- В результате Z-теста пропорций мы получили `p-value = 0.010393282955333764` при `уровне значимости = 0.05`, т.е. шанс получить такие и более выраженные значения при верной нулевой гипотезе `~1%`. Таким образом, по имеющимся данным, можно принять альтернативную гипотезу. **Альтернативная гипотеза:** CR в покупку в группе B *больше*, чем в группе A. Ожидаемый эффект в изменении коверсии был достигнут.